# NHL Scoring First — Analysis

"Scoring first wins games" is one of the most repeated phrases in hockey. Coaches say it before every game. Broadcasters reference it after every first goal. But how true is it, and does *when* you score first matter as much as the goal itself?

This notebook works through three layers of evaluation using 16,500+ regular season NHL games from 2010 to 2023:

1. **Exploratory Data Analysis** — Surface the patterns across period, timing, strength state, home/away, and season
2. **Statistical Testing** — Use chi-square tests to determine whether differences between groups are statistically significant
3. **Logistic Regression Model** — Predict win probability from first goal context and interpret the actual coefficients

Unlike the Random Forest models used in the goalie pull and fight momentum projects, logistic regression produces a real equation with interpretable coefficients — making it easy to quantify exactly how much each factor changes the probability of winning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load the extracted dataset
df = pd.read_csv('NHL_Scoring_First_Data.csv')

# Filter out overtime and shootout first goals for the main analysis
# A first goal in OT always wins the game (ScoreState = 1 ends it),
# so including them inflates win rates and skews the model
df_reg = df[df['First_Goal_Period'] <= 3].copy()

print(f'Total games loaded: {len(df):,}')
print(f'Regulation first goals (periods 1-3): {len(df_reg):,}')
print(f'OT/Shootout first goals excluded: {len(df) - len(df_reg):,}')
print()
print(f'Overall win rate (all games): {df["First_Goal_Team_Won"].mean():.1%}')
print(f'Overall win rate (regulation first goals only): {df_reg["First_Goal_Team_Won"].mean():.1%}')

---
## Part 1 — Exploratory Data Analysis

Before running any tests or models, we visualize the data to understand the patterns. EDA helps us see where the story is most interesting and what questions are worth testing statistically.

In [ ]:
# --- VISUAL 1: Overall Win Rate ---
# The baseline number everyone quotes. How often does the team that scores
# first actually win? We show regulation and all games side by side.

overall_rates = {
    'All Games\n(incl. OT/SO)': df['First_Goal_Team_Won'].mean(),
    'Regulation\nFirst Goals Only': df_reg['First_Goal_Team_Won'].mean(),
    'Baseline\n(50/50 coin flip)': 0.5
}

colors = ['#2ecc71', '#3498db', '#95a5a6']
plt.figure(figsize=(8, 5))
bars = plt.bar(overall_rates.keys(), overall_rates.values(),
               color=colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, overall_rates.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% (no advantage)')
plt.ylim(0, 0.75)
plt.title('Win Rate for Team That Scores First', fontsize=14)
plt.ylabel('Win Rate', fontsize=12)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Overall_Win_Rate.png')
plt.show()

In [ ]:
# --- VISUAL 2: Win Rate by Period ---
# This is the most important breakdown. A first goal in the 1st period
# leaves two full periods of hockey to play. A first goal in the 3rd
# is essentially a go-ahead goal with limited time to respond.
# The question is whether the data reflects that difference.

period_stats = df_reg.groupby('First_Goal_Period_Label')['First_Goal_Team_Won'].agg(
    ['mean', 'count']
).reindex(['Period 1', 'Period 2', 'Period 3'])

fig, ax1 = plt.subplots(figsize=(10, 6))
colors = ['#3498db', '#e67e22', '#e74c3c']
bars = ax1.bar(period_stats.index, period_stats['mean'],
               color=colors, edgecolor='white', linewidth=1.2)
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='50% baseline')
ax1.axhline(df_reg['First_Goal_Team_Won'].mean(), color='green',
            linestyle='--', alpha=0.6, label=f'Overall avg ({df_reg["First_Goal_Team_Won"].mean():.1%})')
for bar, (mean, count) in zip(bars, period_stats.itertuples(index=False)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{mean:.1%}\n(n={count:,})', ha='center', fontsize=11)
ax1.set_ylim(0, 0.85)
ax1.set_title('Win Rate by Period of First Goal', fontsize=14)
ax1.set_ylabel('Win Rate', fontsize=12)
ax1.set_xlabel('Period in Which First Goal Was Scored', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Win_Rate_By_Period.png')
plt.show()

In [ ]:
# --- VISUAL 3: Win Rate by Timing Within Period ---
# Does it matter whether the first goal comes early, middle, or late
# within a period? An early 1st period goal gives maximum time to
# capitalize on momentum. A late 1st period goal sends a team into
# the first intermission trailing.

timing_order = ['Early (0-5 min)', 'Middle (5-15 min)', 'Late (15-20 min)']
timing_stats = df_reg.groupby('First_Goal_Timing')['First_Goal_Team_Won'].agg(
    ['mean', 'count']
).reindex(timing_order)

plt.figure(figsize=(10, 5))
bars = plt.bar(timing_stats.index, timing_stats['mean'],
               color=['#1abc9c', '#3498db', '#9b59b6'], edgecolor='white')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='50% baseline')
plt.axhline(df_reg['First_Goal_Team_Won'].mean(), color='green',
            linestyle='--', alpha=0.6, label=f'Overall avg ({df_reg["First_Goal_Team_Won"].mean():.1%})')
for bar, (mean, count) in zip(bars, timing_stats.itertuples(index=False)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{mean:.1%}\n(n={count:,})', ha='center', fontsize=11)
plt.ylim(0, 0.75)
plt.title('Win Rate by Timing of First Goal Within Period', fontsize=14)
plt.ylabel('Win Rate', fontsize=12)
plt.xlabel('When Within the Period the First Goal Was Scored', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Win_Rate_By_Timing.png')
plt.show()

In [ ]:
# --- VISUAL 4: Win Rate by Strength State ---
# Does it matter whether the first goal was a power play goal,
# even strength, or shorthanded? A shorthanded goal is considered
# a massive momentum swing. Does the data support that?

sit_order = ['Even Strength', 'Power Play', 'Shorthanded']
sit_stats = df_reg[df_reg['First_Goal_Situation'].isin(sit_order)].groupby(
    'First_Goal_Situation'
)['First_Goal_Team_Won'].agg(['mean', 'count']).reindex(sit_order)

plt.figure(figsize=(10, 5))
bars = plt.bar(sit_stats.index, sit_stats['mean'],
               color=['#3498db', '#f39c12', '#e74c3c'], edgecolor='white')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='50% baseline')
plt.axhline(df_reg['First_Goal_Team_Won'].mean(), color='green',
            linestyle='--', alpha=0.6, label=f'Overall avg ({df_reg["First_Goal_Team_Won"].mean():.1%})')
for bar, (mean, count) in zip(bars, sit_stats.itertuples(index=False)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{mean:.1%}\n(n={count:,})', ha='center', fontsize=11)
plt.ylim(0, 0.75)
plt.title('Win Rate by Strength State of First Goal', fontsize=14)
plt.ylabel('Win Rate', fontsize=12)
plt.xlabel('Strength State When First Goal Was Scored', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Win_Rate_By_Situation.png')
plt.show()

In [ ]:
# --- VISUAL 5: Home vs Away ---
# Home teams scoring first should have an advantage due to crowd energy
# and last change privilege. Does the data show a meaningful gap?

home_stats = df_reg.groupby('Home_Team_Scored_First')['First_Goal_Team_Won'].agg(
    ['mean', 'count']
)
home_stats.index = ['Away Team Scored First', 'Home Team Scored First']

plt.figure(figsize=(8, 5))
bars = plt.bar(home_stats.index, home_stats['mean'],
               color=['#e74c3c', '#2ecc71'], edgecolor='white')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label='50% baseline')
for bar, (mean, count) in zip(bars, home_stats.itertuples(index=False)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{mean:.1%}\n(n={count:,})', ha='center', fontsize=12)
plt.ylim(0, 0.75)
plt.title('Win Rate by Which Team Scored First', fontsize=14)
plt.ylabel('Win Rate for the Team That Scored First', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Win_Rate_By_Venue.png')
plt.show()

In [ ]:
# --- VISUAL 6: Win Rate by Season ---
# Has the value of scoring first changed over 14 seasons?
# If the NHL has become a more come-from-behind league (better goaltending,
# faster pace, analytics-driven play), we might expect the win rate to decline.

season_stats = df_reg.groupby('Season')['First_Goal_Team_Won'].mean()

plt.figure(figsize=(11, 5))
plt.plot(season_stats.index, season_stats.values,
         marker='o', linewidth=2.5, color='steelblue')
plt.fill_between(season_stats.index, season_stats.values, alpha=0.15, color='steelblue')
plt.axhline(df_reg['First_Goal_Team_Won'].mean(), color='green',
            linestyle='--', alpha=0.7, label=f'14-season avg ({df_reg["First_Goal_Team_Won"].mean():.1%})')
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% baseline')
plt.title('Win Rate for Team Scoring First by Season (2010–2023)', fontsize=14)
plt.xlabel('Season', fontsize=12)
plt.ylabel('Win Rate', fontsize=12)
plt.ylim(0.45, 0.70)
plt.legend(fontsize=10)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Win_Rate_By_Season.png')
plt.show()

In [ ]:
# --- VISUAL 7: Period x Timing Heatmap ---
# Combining period and timing gives us the most granular view of
# when a first goal matters most. A late 3rd period goal vs. an early
# 1st period goal are very different situations — this heatmap shows
# the full matrix of win rates across every combination.

timing_order = ['Early (0-5 min)', 'Middle (5-15 min)', 'Late (15-20 min)']
period_order = ['Period 1', 'Period 2', 'Period 3']

heatmap_data = df_reg.groupby(
    ['First_Goal_Period_Label', 'First_Goal_Timing']
)['First_Goal_Team_Won'].mean().unstack()
heatmap_data = heatmap_data.reindex(index=period_order, columns=timing_order)

plt.figure(figsize=(10, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.1%', cmap='RdYlGn',
            vmin=0.5, vmax=0.85, linewidths=0.5, cbar_kws={'label': 'Win Rate'})
plt.title('Win Rate Heatmap: Period vs. Timing of First Goal', fontsize=14)
plt.xlabel('When Within the Period', fontsize=12)
plt.ylabel('Period', fontsize=12)
plt.tight_layout()
plt.savefig('outputs/Win_Rate_Heatmap.png')
plt.show()

---
## Part 2 — Statistical Testing: Chi-Square Test

EDA shows us patterns but cannot confirm whether they are statistically meaningful. For a binary outcome like win/loss, the appropriate test is a **chi-square test of independence** rather than a t-test. Chi-square tests whether the observed distribution of wins and losses across groups differs significantly from what we would expect if the groups had no effect.

We test three key comparisons:
- Does the period of the first goal significantly affect win rate?
- Does timing within the period significantly affect win rate?
- Does home/away status significantly affect win rate?

A p-value below 0.05 in each case means the difference between groups is statistically significant and not due to random chance.

In [ ]:
# --- CHI-SQUARE TESTS ---

def run_chi_square(df, group_col, target_col='First_Goal_Team_Won', label=''):
    contingency = pd.crosstab(df[group_col], df[target_col])
    chi2, p, dof, expected = stats.chi2_contingency(contingency)
    significance = '*** SIGNIFICANT' if p < 0.05 else 'not significant'
    print(f'{label}')
    print(f'  Chi-Square Statistic: {chi2:.4f}')
    print(f'  Degrees of Freedom:   {dof}')
    print(f'  P-Value:              {p:.6f}  →  {significance}')
    print()

print('=' * 55)
print('CHI-SQUARE TEST RESULTS')
print('=' * 55)
print()

run_chi_square(df_reg, 'First_Goal_Period_Label',
               label='Does PERIOD of first goal affect win rate?')

run_chi_square(df_reg, 'First_Goal_Timing',
               label='Does TIMING within period affect win rate?')

run_chi_square(df_reg, 'Home_Team_Scored_First',
               label='Does HOME/AWAY status affect win rate?')

run_chi_square(
    df_reg[df_reg['First_Goal_Situation'].isin(['Even Strength','Power Play','Shorthanded'])],
    'First_Goal_Situation',
    label='Does STRENGTH STATE of first goal affect win rate?'
)

print('=' * 55)

---
## Part 3 — Logistic Regression Model

The chi-square tests tell us *whether* each factor is significant. The logistic regression tells us *how much* each factor changes the probability of winning, all else being equal.

Logistic regression was chosen here specifically because it produces interpretable coefficients — unlike Random Forest, we can write out the exact equation and explain what each variable contributes. The output probability is calculated as:

**P(Win) = 1 / (1 + e^-(b0 + b1*Period + b2*Time + b3*Home + ...))**

Each coefficient tells us the log-odds change associated with a one-unit increase in that feature. We convert these to odds ratios for easier interpretation.

In [ ]:
# --- FEATURE PREPARATION ---

model_df = df_reg.copy()

# Encode period as dummy variables (Period 1 is the reference category)
period_dummies = pd.get_dummies(model_df['First_Goal_Period_Label'],
                                 prefix='Period', drop_first=True)

# Encode timing (Early is reference)
timing_dummies = pd.get_dummies(model_df['First_Goal_Timing'],
                                 prefix='Timing', drop_first=True)

# Encode strength state (Even Strength is reference)
sit_df = model_df[model_df['First_Goal_Situation'].isin(
    ['Even Strength', 'Power Play', 'Shorthanded']
)].copy()
sit_dummies = pd.get_dummies(sit_df['First_Goal_Situation'],
                              prefix='Sit', drop_first=True)

model_df = pd.concat([model_df, period_dummies, timing_dummies], axis=1)
model_df = model_df.loc[sit_df.index]
model_df = pd.concat([model_df, sit_dummies], axis=1)

feature_cols = (
    list(period_dummies.columns) +
    list(timing_dummies.columns) +
    list(sit_dummies.columns) +
    ['Home_Team_Scored_First', 'Went_To_OT']
)

X = model_df[feature_cols].fillna(0)
y = model_df['First_Goal_Team_Won']

# Scale features for stable coefficient estimation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f'Model dataset size: {len(model_df):,} games')
print(f'Features: {feature_cols}')

In [ ]:
# --- FIT LOGISTIC REGRESSION ---

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train, y_train)

y_pred = log_reg.predict(X_test)
y_prob = log_reg.predict_proba(X_test)[:, 1]

# Cross-validated AUC score
cv_auc = cross_val_score(log_reg, X_scaled, y, cv=5, scoring='roc_auc').mean()

print('=' * 50)
print('LOGISTIC REGRESSION MODEL RESULTS')
print('=' * 50)
print()
print(classification_report(y_test, y_pred,
      target_names=['Did Not Win', 'Won']))
print(f'ROC-AUC Score (test set):        {roc_auc_score(y_test, y_prob):.4f}')
print(f'ROC-AUC Score (5-fold CV avg):   {cv_auc:.4f}')
print('=' * 50)

In [ ]:
# --- COEFFICIENT INTERPRETATION ---
# Coefficients from logistic regression are in log-odds.
# Converting to odds ratios (e^coefficient) makes them easier to interpret:
# An odds ratio > 1 means that feature increases the probability of winning.
# An odds ratio < 1 means it decreases the probability of winning.

coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': log_reg.coef_[0],
    'Odds_Ratio': np.exp(log_reg.coef_[0])
}).sort_values('Coefficient', ascending=False)

print('=' * 65)
print('COEFFICIENT TABLE')
print('Odds Ratio > 1 = increases win probability')
print('Odds Ratio < 1 = decreases win probability')
print('=' * 65)
print(coef_df.to_string(index=False))
print()
print(f'Intercept (log-odds baseline): {log_reg.intercept_[0]:.4f}')

In [ ]:
# --- VISUAL: Coefficient Plot ---
# Visualizing the coefficients makes it easy to see which factors
# have the strongest positive or negative effect on win probability.

coef_plot = coef_df.sort_values('Coefficient')
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in coef_plot['Coefficient']]

plt.figure(figsize=(10, 6))
plt.barh(coef_plot['Feature'], coef_plot['Coefficient'],
         color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression Coefficients\n(Positive = Increases Win Probability)', fontsize=14)
plt.xlabel('Coefficient (Log-Odds)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/Logistic_Regression_Coefficients.png')
plt.show()

In [ ]:
# --- VISUAL: ROC Curve ---
# The ROC curve shows the trade-off between sensitivity and specificity
# across all possible classification thresholds.
# AUC (Area Under Curve) of 0.5 = random guessing; 1.0 = perfect prediction.

fpr, tpr, _ = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='steelblue', linewidth=2.5,
         label=f'Logistic Regression (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--',
         linewidth=1.2, label='Random Baseline (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Scoring First Win Prediction', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('outputs/ROC_Curve.png')
plt.show()

In [ ]:
# --- WIN PROBABILITY CALCULATOR ---
# Using the fitted model to predict win probability for any scenario.
# This is the interactive equivalent of the goalie pull decision auditor —
# input a game situation and get back the predicted probability of winning
# for the team that just scored the first goal.

def predict_win_probability(period, timing, situation, home_scored_first, went_to_ot=0):
    """
    Predict the win probability for the team that scored first.

    period: 'Period 1', 'Period 2', or 'Period 3'
    timing: 'Early (0-5 min)', 'Middle (5-15 min)', or 'Late (15-20 min)'
    situation: 'Even Strength', 'Power Play', or 'Shorthanded'
    home_scored_first: 1 if home team scored first, 0 if away team
    went_to_ot: 1 if game went to overtime, 0 if not (usually set to 0 for live prediction)
    """
    input_dict = {col: 0 for col in feature_cols}

    if period == 'Period 2':
        input_dict['Period_Period 2'] = 1
    elif period == 'Period 3':
        input_dict['Period_Period 3'] = 1

    if timing == 'Late (15-20 min)':
        col = [c for c in feature_cols if 'Late' in c]
        if col: input_dict[col[0]] = 1
    elif timing == 'Middle (5-15 min)':
        col = [c for c in feature_cols if 'Middle' in c]
        if col: input_dict[col[0]] = 1

    if situation == 'Power Play':
        col = [c for c in feature_cols if 'Power' in c]
        if col: input_dict[col[0]] = 1
    elif situation == 'Shorthanded':
        col = [c for c in feature_cols if 'Short' in c]
        if col: input_dict[col[0]] = 1

    input_dict['Home_Team_Scored_First'] = home_scored_first
    input_dict['Went_To_OT'] = went_to_ot

    input_arr = np.array([[input_dict[col] for col in feature_cols]])
    input_scaled = scaler.transform(input_arr)
    prob = log_reg.predict_proba(input_scaled)[0][1]

    print(f'Scenario: {situation} goal in {timing} of {period} | Home scored first: {bool(home_scored_first)}')
    print(f'Predicted Win Probability: {prob:.1%}')
    return prob

print('EXAMPLE SCENARIOS')
print('=' * 55)
predict_win_probability('Period 1', 'Early (0-5 min)', 'Even Strength', 1)
print()
predict_win_probability('Period 1', 'Early (0-5 min)', 'Even Strength', 0)
print()
predict_win_probability('Period 3', 'Late (15-20 min)', 'Even Strength', 1)
print()
predict_win_probability('Period 2', 'Middle (5-15 min)', 'Power Play', 0)
print()
predict_win_probability('Period 1', 'Early (0-5 min)', 'Shorthanded', 1)

---
## Summary of Findings

Fill this in after running the notebook with your full dataset. Key things to document:

**The cliché is true — but weaker than advertised**
- What was the overall win rate for the team that scored first?
- How does that compare to what people assume?

**Period matters far more than most realize**
- What were the win rates for 1st, 2nd, and 3rd period first goals?
- Were those differences statistically significant?

**What surprised you?**
- Did timing within the period matter?
- Did strength state change win rates meaningfully?
- Did home/away make a difference?

**What the model found**
- Which features had the strongest coefficients?
- What was the ROC-AUC score and what does it tell you?
- What win probability does the calculator predict for the best vs. worst case scenarios?